In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binned_statistic
from matplotlib import rc

from soundspeed import sound_speed_from_ctd, is_inside_Dotson_cavity, sound_speed_in_cavity

rc('font', size=7)
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif'] # Arial as first choice

fig_width = 5.5 
fig_height = 2.5

In [ ]:
sound_speed_all = sound_speed_from_ctd(filename = 'data/auxiliary/CTD/CTD_NBP2202_*.txt')
inside_cavity = is_inside_Dotson_cavity(sound_speed_all)

Compute mean profile in the ice shelf cavity

In [ ]:
bin_size = 10
p = sound_speed_all.where(inside_cavity).p
bins = np.arange(np.min(p),np.max(p),bin_size)
res  = binned_statistic(p,
                        sound_speed_all.where(inside_cavity).c,
                        statistic='mean', 
                        bins= bins)
mean_profile  = pd.DataFrame({'p' : (bins[:-1]+bins[1:])/2,
                              'c' : res.statistic})

Compute fit

In [ ]:
profile = sound_speed_in_cavity(make_plot=False)

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(fig_width/2, fig_height*1.5))

# Open water
ax.scatter(sound_speed_all.where(~inside_cavity).c, 
           sound_speed_all.where(~inside_cavity).p, 
           s=1, c='lightgray', 
           label='open water')

# In cavity
ax.scatter(sound_speed_all.where(inside_cavity).c, 
           sound_speed_all.where(inside_cavity).p, 
           s=1, c='cornflowerblue',
           label='cavity')

# Mean profile
ax.plot(mean_profile.c,
        mean_profile.p,
        'k--', linewidth=1,
        label = 'mean in cavity')

# Fit
ps = np.linspace(0, np.max(sound_speed_all.p), 100)
ax.plot(profile(ps), ps,
       'k', 
        label = 'logarithmic fit')

ax.grid(alpha=0.1)
lgnd = ax.legend(loc='lower left')

# Increase size of scatter points in legend
for handle in lgnd.legend_handles[:2]:
    handle.set_sizes([30.0])

ax.invert_yaxis()
ax.set_xlabel('Sound speed (m/s)')
ax.set_ylabel('Pressure (dbar)')

plt.savefig('figures/fig4.png', bbox_inches = 'tight', dpi=600)